# Sales Forecast AI - Exploratory Data Analysis

This notebook covers the initial **Exploratory Data Analysis (EDA)** of our store sales forecasting dataset. The goal is to understand the distribution, seasonality, store-wise patterns, item popularity, and correlation structure to inform our feature engineering pipeline.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in the system path to import data pipeline utils
sys.path.append(os.path.abspath("../src"))

# Setup style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Load and Inspect Raw Data

In [ ]:
from config.paths import RAW_DATA_DIR

dataset_path = RAW_DATA_DIR / "train.csv"
print(f"Loading dataset from: {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Dataset shape: {df.shape}")

In [ ]:
# Preview first few rows
df.head()

In [ ]:
# General Info
df.info()

In [ ]:
# Missing Values
print("Missing Values:\n", df.isnull().sum())

In [ ]:
# Number of unique stores and items
print(f"Number of Unique Stores: {df['store'].nunique()}")
print(f"Number of Unique Items : {df['item'].nunique()}")

## 2. Target Variable Distribution (`sales`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
sns.histplot(df["sales"], kde=True, bins=50, ax=axes[0], color="royalblue")
axes[0].set_title("Distribution of Sales")
axes[0].set_xlabel("Sales")

# Boxplot
sns.boxplot(x=df["sales"], ax=axes[1], color="lightcoral")
axes[1].set_title("Boxplot of Sales (Outlier Detection)")
axes[1].set_xlabel("Sales")

plt.tight_layout()
plt.show()

In [ ]:
# Describe sales distribution
df["sales"].describe()

## 3. Time Series Trends and Seasonality

Let's parse the `date` column and explore seasonality across years, months, and days of the week.

In [ ]:
# Parse Date
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Aggregate daily sales across all stores and items
daily_sales = df.groupby("date")["sales"].sum().reset_index()

plt.figure(figsize=(14, 6))
plt.plot(daily_sales["date"], daily_sales["sales"], color="teal", alpha=0.8)
plt.title("Total Daily Sales (All Stores & Items)")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.show()

In [ ]:
# Extract calendar components
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek
df["day_name"] = df["date"].dt.day_name()

# 1. Yearly Trend
yearly_avg = df.groupby("year")["sales"].mean().reset_index()
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.barplot(data=yearly_avg, x="year", y="sales", ax=axes[0], palette="Blues_d")
axes[0].set_title("Average Sales by Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Average Sales")

# 2. Monthly Seasonality (Annual Pattern)
monthly_avg = df.groupby("month")["sales"].mean().reset_index()
sns.lineplot(data=monthly_avg, x="month", y="sales", marker="o", ax=axes[1], color="darkorange", linewidth=2.5)
axes[1].set_title("Monthly Sales Seasonality")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Average Sales")
axes[1].set_xticks(range(1, 13))

# 3. Weekly Seasonality (Day-of-Week Pattern)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekly_avg = df.groupby("day_name")["sales"].mean().reindex(day_order).reset_index()
sns.barplot(data=weekly_avg, x="day_name", y="sales", ax=axes[2], palette="viridis")
axes[2].set_title("Weekly Sales Seasonality")
axes[2].set_xlabel("Day of Week")
axes[2].set_ylabel("Average Sales")
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Entity Analysis: Stores and Items

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Store-wise sales distribution
store_sales = df.groupby("store")["sales"].mean().reset_index()
sns.barplot(data=store_sales, x="store", y="sales", ax=axes[0], palette="rocket")
axes[0].set_title("Average Sales per Store")
axes[0].set_xlabel("Store ID")
axes[0].set_ylabel("Average Sales")

# Item-wise sales distribution (Top 15 items)
item_sales = df.groupby("item")["sales"].mean().reset_index().sort_values(by="sales", ascending=False)
sns.barplot(data=item_sales.head(15), x="item", y="sales", ax=axes[1], palette="mako", order=item_sales.head(15)["item"])
axes[1].set_title("Top 15 Best Selling Items (Average Sales)")
axes[1].set_xlabel("Item ID")
axes[1].set_ylabel("Average Sales")

plt.tight_layout()
plt.show()

## 5. Lag and Autocorrelation Analysis

Understanding historical dependencies (lags) is critical for feature engineering.

In [ ]:
# Focus on a single store/item combination for clear autocorrelation signal
single_series = df[(df["store"] == 1) & (df["item"] == 1)].sort_values("date").copy()

# Create lag columns manually for analysis
for l in [1, 2, 3, 7, 14, 28]:
    single_series[f"lag_{l}"] = single_series["sales"].shift(l)

# Calculate correlation
lag_cols = ["sales"] + [f"lag_{l}" for l in [1, 2, 3, 7, 14, 28]]
corr = single_series[lag_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".3f", vmin=0, vmax=1)
plt.title("Autocorrelation of Sales (Lags for Store 1, Item 1)")
plt.show()

### Conclusion of EDA
1. **Trend**: Sales have a clear upward year-over-year trend.
2. **Seasonality**: There is a strong annual pattern peaking in mid-summer (July) and a weekly cycle peaking on weekends (Friday - Sunday).
3. **Store/Item Variance**: Significant variations exist across stores (some perform double the volume of others) and items, highlighting the importance of grouping features (like lag and rolling windows) by store and item entity.
4. **Autocorrelation**: Strong positive correlation with recent lags (especially lag 1 and lag 7), verifying that time-series lags will be highly informative features for our machine learning models.